In [ ]:
#| default_exp responses

# Responses facade

> Provider-independent Responses resources and streaming over FastLLM

A Responses request is one model turn. This module translates the standard wire format to FastLLM messages, chooses provider-id continuation or canonical replay, and translates the resulting `Completion` back to Responses resources and events. It deliberately does not store responses or execute requested tools.

## Imports

In [ ]:
import asyncio, copy, json, secrets, time
from fastcore.utils import *
from fastcore.funccall import call_func_async, get_schema
from fasttransport.errors import APIError

from fastllm.acomplete import acomplete
from fastllm.streaming import Status
from aidialog.msg_parts import Msg, Text, ToolUse, ToolResult, InputImage, InputFile, Completion, tool_text, msg2dict, dict2msg

In [ ]:
from fastcore.test import *
from cachy import enable_cachy

## Input items

`response_input` converts a plain prompt or standard input-item array to canonical messages. System and developer messages become provider instructions; adjacent function outputs remain one tool message so a parallel batch reaches the next model turn together.

In [ ]:
class ResponsesError(Exception):
    "A request error with Responses-compatible metadata."
    def __init__(self, message, status=400, code='invalid_request_error', param=None):
        super().__init__(message)
        store_attr()

Responses content may use a plain string or typed blocks. `_msg_parts` preserves text, image, and file inputs as the corresponding canonical FastLLM parts, so provider adapters never need to understand the Responses wire shape.

In [ ]:
def _msg_parts(content):
    if isinstance(content, str): return [Text(content)]
    parts = []
    for o in content:
        typ = o.get('type')
        if typ in ('input_text', 'output_text', 'text'): parts.append(Text(o.get('text', '')))
        elif typ == 'input_image': parts.append(InputImage(o.get('image_url')))
        elif typ == 'input_file': parts.append(InputFile(o.get('file_url') or o.get('file_data')))
        else: raise ResponsesError(f'Unsupported message content type: {typ}', param='input')
    return parts

In [ ]:
text_block = dict(type='input_text', text='Describe this image.')
image_block = dict(type='input_image', image_url='data:image/png;base64,AAAA')
parts = _msg_parts([text_block, image_block])
test_eq([type(o) for o in parts], [Text, InputImage])
parts

[Text(raw=None, cache_control=None, text='Describe this image.', citations=None),
 InputImage(raw=None, cache_control=None, text='data:image/png;base64,AAAA', mime=None)]

Developer and system items become provider instructions rather than ordinary history. User content remains a canonical message, including its typed parts.

In [ ]:
def _text_content(content):
    if isinstance(content, str): return content
    return ''.join(o.get('text', '') for o in content if o.get('type') in ('input_text', 'output_text', 'text'))

def response_input(inp):
    "Convert Responses input items to canonical messages and instructions."
    if isinstance(inp, str): return [Msg('user', [Text(inp)])], ''
    if not isinstance(inp, list): raise ResponsesError('input must be a string or array', param='input')
    msgs,systems,pending_tools = [],[],[]
    def flush_tools():
        if pending_tools:
            msgs.append(Msg('tool', pending_tools.copy()))
            pending_tools.clear()
    for item in inp:
        typ = item.get('type')
        if typ in ('message', 'easy_input_message') or 'role' in item:
            flush_tools()
            role = item.get('role', 'user')
            if role in ('system', 'developer'): systems.append(_text_content(item.get('content', '')))
            else: msgs.append(Msg(role, _msg_parts(item.get('content', ''))))
        elif typ == 'function_call':
            flush_tools()
            args = item.get('arguments') or {}
            if isinstance(args, str):
                try: args = json.loads(args)
                except json.JSONDecodeError: pass
            call = ToolUse(id=item.get('call_id'), name=item.get('name'), arguments=args)
            if msgs and msgs[-1].role == 'assistant': msgs[-1].content.append(call)
            else: msgs.append(Msg('assistant', [call]))
        elif typ == 'function_call_output':
            out = item.get('output', '')
            pending_tools.append(ToolResult(id=item.get('call_id'), name=item.get('name', ''),
                text=out if isinstance(out, str) else _msg_parts(out)))
        else: raise ResponsesError(f'Unsupported input item type: {typ}', param='input')
    flush_tools()
    return msgs,'\n\n'.join(filter(None, systems))

In [ ]:
input_items = [dict(role='developer', content='Answer briefly.'),
    dict(role='user', content=[dict(type='input_text', text='What is shown?')])]
input_msgs,instructions = response_input(input_items)
test_eq((instructions, input_msgs[0].text), ('Answer briefly.', 'What is shown?'))
dict(instructions=instructions, message=input_msgs[0])

{'instructions': 'Answer briefly.',
 'message': Msg(role='user', content=[Text(raw=None, cache_control=None, text='What is shown?', citations=None)])}

A completion with text and a parallel batch of calls arrives as one message item and several `function_call` items. They become a single assistant message, and the batch's outputs a single tool message, so a provider that pairs each call with the tool messages right after it sees the batch whole.


In [ ]:
batch_items = [dict(type='message', role='assistant', content=[dict(type='output_text', text='Computing both.')]),
    dict(type='function_call', call_id='call_a', name='py', arguments='{"code":"2+2"}'),
    dict(type='function_call', call_id='call_b', name='py', arguments='{"code":"3*3"}'),
    dict(type='function_call_output', call_id='call_a', output='4'),
    dict(type='function_call_output', call_id='call_b', output='9')]
batch_msgs,system = response_input(batch_items)
test_eq(system, '')
test_eq([m.role for m in batch_msgs], ['assistant', 'tool'])
test_eq([type(p).__name__ for p in batch_msgs[0].content], ['Text', 'ToolUse', 'ToolUse'])
test_eq([o.id for o in batch_msgs[1].content], ['call_a', 'call_b'])
batch_msgs[0]


**Msg**

- role: `tool`

<contents>

**ToolResult** (`tool_result`)

4

::: details

- raw: `None`
- id: `call_a`
- name: ``
- arguments: `{}`
- server: `False`

:::

**ToolResult** (`tool_result`)

9

::: details

- raw: `None`
- id: `call_b`
- name: ``
- arguments: `{}`
- server: `False`

:::

</contents>

An array output keeps its media: a tool that returned an image arrives as canonical parts, which each provider's tool-result denorm re-emits in its own image format.

In [ ]:
img = 'data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAQAAAAECAIAAAAmkwkpAAAAEElEQVR4nGP8z4AATAxIAAB2+AEAqfB/ugAAAABJRU5ErkJggg=='
media_out = [dict(type='function_call_output', call_id='call_c',
    output=[dict(type='input_text', text='the chart:'), dict(type='input_image', image_url=img)])]
mmsgs,_ = response_input(media_out)
parts = mmsgs[0].content[0].text
test_eq([type(p).__name__ for p in parts], ['Text', 'InputImage'])
test_eq(parts[1].text, img)
mmsgs[0]

**Msg**

- role: `tool`

<contents>

**ToolResult** (`tool_result`)

[Text(raw=None, cache_control=None, text='the chart:', citations=None), InputImage(raw=None, cache_control=None, text='data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAQAAAAECAIAAAAmkwkpAAAAEElEQVR4nGP8z4AATAxIAAB2+AEAqfB/ugAAAABJRU5ErkJggg==', mime=None)]

::: details

- raw: `None`
- id: `call_c`
- name: ``
- arguments: `{}`
- server: `False`

:::

</contents>

## Response resources

FastLLM completions become ordinary assistant-message and function-call items. Call ids are normalized before the canonical history is retained, so a later tool result always refers to the same id the client saw.

In [ ]:
def _new_id(prefix): return f'{prefix}_{secrets.token_hex(16)}'

def normalize_call_ids(comp):
    "Ensure every client tool call has a Responses call id."
    for tc in comp.tool_calls:
        if not tc.id: tc.id = _new_id('call')
    return comp

A normalized `Completion` can contain both assistant text and client-executed calls. `response_output` keeps those as separate Responses items, after `normalize_call_ids` supplies an id when an upstream omitted one.

In [ ]:
def _arguments(args): return args if isinstance(args, str) else json.dumps(args or {}, separators=(',', ':'))

def response_output(comp):
    "Convert a Completion to Responses output items."
    output,msg = [],None
    for part in comp.message.content:
        if isinstance(part, Text):
            if msg is None:
                content = dict(type='output_text', text=part.text or '', annotations=list(part.citations or []))
                msg = dict(id=getattr(part, '_response_item_id', None) or _new_id('msg'), type='message', status='completed',
                    role='assistant', content=[content])
                output.append(msg)
            else:
                msg['content'][0]['text'] += part.text or ''
                msg['content'][0]['annotations'].extend(part.citations or [])
        elif isinstance(part, ToolUse) and not part.server:
            output.append(dict(id=getattr(part, '_response_item_id', None) or _new_id('fc'), type='function_call', status='completed',
                call_id=part.id, name=part.name, arguments=_arguments(part.arguments)))
    return output

In [ ]:
sample_usage = dict2obj(dict(prompt_tokens=12, completion_tokens=4, total_tokens=16,
    cached_tokens=3, cache_creation_tokens=2, reasoning_tokens=1))
sample_parts = [Text('Done.'), ToolUse(name='py', arguments={'code':'2+2'})]
sample_comp = Completion('demo', Msg('assistant', sample_parts), 'stop', sample_usage)
normalize_call_ids(sample_comp)
sample_output = response_output(sample_comp)
test_eq([o['type'] for o in sample_output], ['message', 'function_call'])
sample_output

[{'id': 'msg_3b9a651d49197423f43290ffa3186f3c',
  'type': 'message',
  'status': 'completed',
  'role': 'assistant',
  'content': [{'type': 'output_text', 'text': 'Done.', 'annotations': []}]},
 {'id': 'fc_b5a04d0296cc806efccbc1e606bfd77c',
  'type': 'function_call',
  'status': 'completed',
  'call_id': 'call_276a5102b44a9ebaafbda2b129f42175',
  'name': 'py',
  'arguments': '{"code":"2+2"}'}]

Usage is already provider-normalized by FastLLM. `response_usage` only gives those totals their standard Responses field names, retaining cache writes and reasoning tokens in the detail objects.

In [ ]:
def response_usage(usage):
    "Convert normalized usage to Responses usage fields."
    if usage is None: return None
    result = dict(input_tokens=usage.prompt_tokens, output_tokens=usage.completion_tokens, total_tokens=usage.total_tokens)
    result['input_tokens_details'] = {'cached_tokens':usage.cached_tokens, 'cache_write_tokens':usage.cache_creation_tokens}
    result['output_tokens_details'] = {'reasoning_tokens':usage.reasoning_tokens}
    return result

In [ ]:
sample_response_usage = response_usage(sample_usage)
test_eq(sample_response_usage['input_tokens_details'], dict(cached_tokens=3, cache_write_tokens=2))
sample_response_usage

{'input_tokens': 12,
 'output_tokens': 4,
 'total_tokens': 16,
 'input_tokens_details': {'cached_tokens': 3, 'cache_write_tokens': 2},
 'output_tokens_details': {'reasoning_tokens': 1}}

Pending and terminal resources must echo the same request controls. `_response_base` builds that shared identity and request metadata before either lifecycle state adds its own fields.

In [ ]:
def _response_base(rid, body, previous_response_id=None, created_at=None):
    result = dict(id=rid, object='response', created_at=int(created_at or time.time()), instructions=body.get('instructions'), model=body['model'])
    result.update(parallel_tool_calls=body.get('parallel_tool_calls', True), previous_response_id=previous_response_id,
        reasoning=body.get('reasoning'), store=True, temperature=body.get('temperature'))
    result.update(text=body.get('text', {'format':{'type':'text'}}), tool_choice=body.get('tool_choice', 'auto'),
        tools=body.get('tools', []), top_p=body.get('top_p'), metadata=body.get('metadata', {}))
    return result

In [ ]:
resource_body = dict(model='demo/model', input='Hello', parallel_tool_calls=True)
resource_base = _response_base('resp_demo', resource_body, created_at=123)
test_eq((resource_base['id'], resource_base['created_at']), ('resp_demo', 123))
resource_base

{'id': 'resp_demo',
 'object': 'response',
 'created_at': 123,
 'instructions': None,
 'model': 'demo/model',
 'parallel_tool_calls': True,
 'previous_response_id': None,
 'reasoning': None,
 'store': True,
 'temperature': None,
 'text': {'format': {'type': 'text'}},
 'tool_choice': 'auto',
 'tools': [],
 'top_p': None,
 'metadata': {}}

A completed resource combines the echoed request, normalized output items, usage, and terminal status. Hitting the provider's output limit is represented as an incomplete Response rather than a different object shape.

In [ ]:
def response_object(rid, body, comp, previous_response_id=None, created_at=None):
    "Build a completed Responses resource from a Completion."
    incomplete = comp.finish_reason == 'length'
    output = response_output(comp)
    result = _response_base(rid, body, previous_response_id, created_at)
    result.update(completed_at=int(time.time()), status='incomplete' if incomplete else 'completed', error=None,
        incomplete_details={'reason':'max_output_tokens'} if incomplete else None, output=output, usage=response_usage(comp.usage))
    result['output_text'] = ''.join(c['text'] for o in output if o['type']=='message' for c in o['content'] if c['type']=='output_text')
    return result

In [ ]:
sample_response = response_object('resp_demo', resource_body, sample_comp, created_at=123)
test_eq((sample_response['status'], sample_response['output_text']), ('completed', 'Done.'))
dict(status=sample_response['status'], output=sample_response['output'], usage=sample_response['usage'])

{'status': 'completed',
 'output': [{'id': 'msg_9700dab1e1e5bba3f17114120d3dfa0c',
   'type': 'message',
   'status': 'completed',
   'role': 'assistant',
   'content': [{'type': 'output_text', 'text': 'Done.', 'annotations': []}]},
  {'id': 'fc_f2c84c266500fc1ac155176c0b5f27d8',
   'type': 'function_call',
   'status': 'completed',
   'call_id': 'call_276a5102b44a9ebaafbda2b129f42175',
   'name': 'py',
   'arguments': '{"code":"2+2"}'}],
 'usage': {'input_tokens': 12,
  'output_tokens': 4,
  'total_tokens': 16,
  'input_tokens_details': {'cached_tokens': 3, 'cache_write_tokens': 2},
  'output_tokens_details': {'reasoning_tokens': 1}}}

## One Responses turn

`ResponseState` is the complete provider-independent continuation value. A host may store it under any ownership and expiry policy. When a provider id exists, the next turn sends only new items; otherwise it replays canonical history.

In [ ]:
class ResponseState(BasicRepr):
    "Canonical history and optional provider continuation metadata."
    def __init__(self, id, model, history, provider_response_id=None): store_attr()

In [ ]:
base_body = dict(model='openai/gpt-5.6-luna', input='Use py.', tools=[])
prior_msg = Msg('user', [Text('Earlier context')])
replay_state = ResponseState('resp_replay', base_body['model'], (prior_msg,))
replay_state

ResponseState(id='resp_replay', model='openai/gpt-5.6-luna', history=(Msg(role='user', content=[Text(raw=None, cache_control=None, text='Earlier context', citations=None)]),), provider_response_id=None, provider_response_reusable=True)

A state converts to a JSON-ready dict and back, `raw` included, so a host can keep it wherever its process boundary demands.

In [ ]:
#| export
@patch
def to_dict(self:ResponseState):
    "This state as a JSON-ready dict"
    return dict(id=self.id, model=self.model, history=[msg2dict(m) for m in self.history], provider_response_id=self.provider_response_id)

@patch(cls_method=True)
def from_dict(cls:ResponseState, d):
    "The state `to_dict` produced `d` from"
    return cls(d['id'], d['model'], tuple(dict2msg(m) for m in d['history']), d['provider_response_id'])

In [ ]:
test_eq(ResponseState.from_dict(json.loads(json.dumps(replay_state.to_dict()))).history, replay_state.history)
replay_state.to_dict()

`ResponseStore` keeps states for `ttl` seconds under their public id, together with whatever `meta` the host records about them, such as who may continue one. The default holds them in this process. `dump` and `load` are the two methods that touch storage, and they see only JSON-ready dicts, so a store shared between processes overrides just those two, with Redis's `SET ... EX` and `GET` for instance.

In [ ]:
#| export
class ResponseStore:
    "Continuation states kept for `ttl` seconds; override `dump` and `load` to keep them outside this process"
    def __init__(self, ttl=3600): self.ttl,self.items = ttl,{}

    async def dump(self, rid, data):
        "Keep JSON-ready `data` under `rid` for `ttl` seconds"
        now = time.time()
        self.items = {k:v for k,v in self.items.items() if v[0] > now}
        self.items[rid] = (now+self.ttl, data)

    async def load(self, rid):
        "The data kept under `rid`, or `None` once it has expired"
        exp,data = self.items.get(rid, (0, None))
        return data if exp > time.time() else None

    async def put(self, state, **meta):
        "Store `state` with the host's `meta` about it"
        await self.dump(state.id, dict(state=state.to_dict(), **meta))

    async def get(self, rid):
        "The `(state, meta)` stored under `rid`, or `None`"
        data = await self.load(rid)
        if data is None: return None
        return ResponseState.from_dict(data['state']),{k:v for k,v in data.items() if k != 'state'}

In [ ]:
store = ResponseStore(ttl=60)
await store.put(replay_state, owner='alice')
state,meta = await store.get('resp_replay')
test_eq((state.history, meta), (replay_state.history, dict(owner='alice')))
test_eq((await store.get('resp_replay'))[1], meta)
test_is(await store.get('resp_missing'), None)
await ResponseStore(ttl=0).put(replay_state)
test_is(await ResponseStore(ttl=0).get('resp_replay'), None)
meta

An external store sees a state only as text. A subclass that keeps JSON strings shows the round trip that Redis or a database row would make.

In [ ]:
class JsonStore(ResponseStore):
    def __init__(self): self.strs = {}
    async def dump(self, rid, data): self.strs[rid] = json.dumps(data)
    async def load(self, rid): return json.loads(self.strs[rid]) if rid in self.strs else None

jstore = JsonStore()
await jstore.put(replay_state, owner='alice')
state,meta = await jstore.get('resp_replay')
test_eq(state.history, replay_state.history)
jstore.strs['resp_replay']

`ResponseTurn` is the ephemeral plan for one provider call: the full canonical history is retained for the next public state, while `provider_messages` contains only what this particular upstream must receive.

In [ ]:
class ResponseTurn(BasicRepr):
    "A prepared Responses request for one provider call."
    def __init__(self, id, created_at, body, history, provider_messages, system='', previous=None, server_tools=None, call_kwargs=None):
        store_attr()
        if server_tools is None: self.server_tools = {}
        if call_kwargs is None: self.call_kwargs = {}

    @property
    def provider_previous_id(self): return self.previous.provider_response_id if self.previous else None


`AsyncResponses.prepare` applies that distinction. Its `server_tools` are functions the host runs itself during the request (see `server_results` below); they are advertised to the provider beside the client's tools. It validates the request, translates new input, combines instructions, and chooses incremental provider input only when the prior state has a usable provider continuation id. Any further keyword arguments ride on the turn as `call_kwargs`, reaching every provider call the turn makes: a host serving many users passes each one's own `oauth_token` this way.

In [ ]:
class AsyncResponses:
    "Run individual Responses turns over FastLLM."
    def __init__(self): self.tasks = set()   # served turns still running
    def prepare(self, body, previous=None, server_tools=None, **kwargs):
        if not isinstance(body, dict): raise ResponsesError('Request body must be an object')
        model = body.get('model')
        if not model: raise ResponsesError('model is required', param='model')
        if 'input' not in body: raise ResponsesError('input is required', param='input')
        if previous and previous.model != model: raise ResponsesError('model must match the previous response', param='model')
        new_msgs,input_system = response_input(body['input'])
        history = (*previous.history, *new_msgs) if previous else tuple(new_msgs)
        names = {p.id: p.name for m in history for p in m.content if isinstance(p, ToolUse)}
        for m in new_msgs:
            for p in m.content:
                if isinstance(p, ToolResult) and not p.name: p.name = names.get(p.id, '')
        instructions = '\n\n'.join(filter(None, [body.get('instructions', ''), input_system]))
        provider_messages = new_msgs if previous and previous.provider_response_id else list(history)
        return ResponseTurn(_new_id('resp'), int(time.time()), body, history, provider_messages, instructions, previous,
            {f.__name__: f for f in server_tools or []}, kwargs)


The same continuation state produces either an incremental provider request or a replay. The distinction is visible in the prepared turn rather than hidden in the proxy route.

In [ ]:
provider_state = ResponseState('resp_provider', base_body['model'], (prior_msg,), 'upstream_123')
responses = AsyncResponses()
replay_turn = responses.prepare(base_body, replay_state)
provider_turn = responses.prepare(base_body, provider_state)
test_eq(len(replay_turn.provider_messages), 2)
test_eq(len(provider_turn.provider_messages), 1)
(provider_turn.provider_previous_id, [m.text for m in replay_turn.provider_messages])
user_turn = responses.prepare(base_body, api_key='per-user-credential')
test_eq(user_turn.call_kwargs, dict(api_key='per-user-credential'))

('upstream_123', ['Earlier context', 'Use py.'])

`call` always starts a provider stream, waits for its first chunk so a rejected request raises here rather than mid-stream, and forwards only the Responses controls understood by FastLLM. That includes the `cache_idxs`/`ttl` body extension: a serving host that assembled the input knows where the stable prefix ends, and transports whose provider needs explicit cache breakpoints (Anthropic) apply them, while the rest ignore them. `web_search_options` rides the same way: the host decides which turns get provider-native search, and each transport denorms it into its provider's search tool.

In [ ]:
@patch
async def call(self:AsyncResponses, turn):
    "Start the provider stream for a prepared turn, having received its first chunk; the turn's `call_kwargs` reach `acomplete`"
    body = turn.body
    reasoning = body.get('reasoning') or {}
    tools = list(body.get('tools') or []) + [dict(type='function', **get_schema(f, pname='parameters')) for f in turn.server_tools.values()]
    kwargs = dict(previous_response_id=turn.provider_previous_id, system=turn.system or None, tools=tools or None)
    kwargs.update(tool_choice=body.get('tool_choice'), parallel_tool_calls=body.get('parallel_tool_calls', True),
        reasoning_effort=reasoning.get('effort'), max_tokens=body.get('max_output_tokens'), temperature=body.get('temperature'),
        cache_idxs=body.get('cache_idxs'), ttl=body.get('ttl'), web_search_options=body.get('web_search_options'))
    stream = await acomplete(turn.provider_messages, model=body['model'], stream=True, **kwargs, **turn.call_kwargs)
    first = await anext(stream)
    async def _rest():
        yield first
        async for o in stream: yield o
    return _rest()


After the provider finishes, `state` retains the assistant message beside the request history, followed by any server tool results that answered it, and records any continuation handle exposed by that transport. The host can then persist this value under its own public response id policy.

In [ ]:
@patch
def state(self:AsyncResponses, turn, comp, results=()):
    "Build the next continuation state; `results` are server tool results answering `comp`'s calls."
    normalize_call_ids(comp)
    history = (*turn.history, comp.message, *([Msg('tool', list(results))] if results else []))
    return ResponseState(turn.id, turn.body['model'], history, comp.response_id)


A server tool is a function the host runs inside the request rather than handing back to the client: the provider sees it as one more function tool, the client never sees its calls. `server_results` runs every server tool call in a completion, and `server_turn` prepares the provider call that answers them, exactly as a client continuation would, keeping the public response id.

In [ ]:
#| export
@patch
async def server_results(self:AsyncResponses, turn, comp):
    "Run the server tool calls in `comp`, returning their results"
    normalize_call_ids(comp)
    res = []
    for tc in comp.tool_calls:
        if tc.name not in turn.server_tools: continue
        out = await call_func_async(tc.name, tc.arguments, ns=turn.server_tools, raise_on_err=False)
        res.append(ToolResult(id=tc.id, name=tc.name, arguments=tc.arguments, text=tool_text(out)))
    return res

@patch
def server_turn(self:AsyncResponses, turn, comp, results):
    "The follow-up provider turn answering server tool `results`, within the same public response"
    outputs = [dict(type='function_call_output', call_id=r.id, output=r.text) for r in results]
    nxt = self.prepare(turn.body | dict(input=outputs), self.state(turn, comp), turn.server_tools.values(), **turn.call_kwargs)
    nxt.id = turn.id
    return nxt

In [ ]:
def lookup(word:str)->str: # The glossary entry
    "Look up `word` in the project glossary"
    return {'gorp': 'a trail mix of nuts and dried fruit'}.get(word, 'not in the glossary')
lookup_turn = responses.prepare(base_body, server_tools=[lookup])
lookup_comp = Completion('demo', Msg('assistant', [ToolUse(id='call_1', name='lookup', arguments=dict(word='gorp')), ToolUse(id='call_2', name='py', arguments=dict(code='1'))]), 'tool_calls')
lookup_results = await responses.server_results(lookup_turn, lookup_comp)
test_eq([(r.name, r.text) for r in lookup_results], [('lookup', 'a trail mix of nuts and dried fruit')])
follow = responses.server_turn(lookup_turn, lookup_comp, lookup_results)
test_eq((follow.id, [m.role for m in follow.history]), (lookup_turn.id, ['user', 'assistant', 'tool']))
follow.provider_messages[-1]

## Event streams

Every request is streamed. `_ResponseStream` only tracks the standard event grammar for one inference; it contains no provider, persistence, or billing policy.

In [ ]:
def response_event(event_type, sequence_number, **kwargs):
    "Encode one Responses SSE frame."
    data = {'type':event_type, **kwargs, 'sequence_number':sequence_number}
    return f"event: {event_type}\ndata: {json.dumps(data, separators=(',', ':'))}\n\n"

In [ ]:
#| hide
def _event_types(frames): return [json.loads(frame.split('data: ', 1)[1])['type'] for frame in frames]

In [ ]:
frame = response_event('response.output_text.delta', 0, delta='Hello')
test_eq(_event_types([frame]), ['response.output_text.delta'])
frame

'event: response.output_text.delta\ndata: {"type":"response.output_text.delta","delta":"Hello","sequence_number":0}\n\n'

A stream opens with a complete in-progress Response resource. `pending_response` echoes the request and previous public response id, but has no output or usage until the provider finishes.

In [ ]:
def pending_response(turn):
    "Build the initial in-progress Responses resource."
    previous_id = turn.previous.id if turn.previous else None
    result = _response_base(turn.id, turn.body, previous_id, turn.created_at)
    result.update(status='in_progress', error=None, incomplete_details=None, output=[], usage=None)
    return result

In [ ]:
pending = pending_response(provider_turn)
test_eq((pending['status'], pending['previous_response_id']), ('in_progress', 'resp_provider'))
pending

{'id': 'resp_51c0f18c7e4bad0aa3de14ca57e3c021',
 'object': 'response',
 'created_at': 1788172901,
 'instructions': None,
 'model': 'openai/gpt-5.6-luna',
 'parallel_tool_calls': True,
 'previous_response_id': 'resp_provider',
 'reasoning': None,
 'store': True,
 'temperature': None,
 'text': {'format': {'type': 'text'}},
 'tool_choice': 'auto',
 'tools': [],
 'top_p': None,
 'metadata': {},
 'status': 'in_progress',
 'error': None,
 'incomplete_details': None,
 'output': [],
 'usage': None}

`_ResponseStream` owns only event-local bookkeeping: the next sequence number, output indexes, accumulated text, and stable item ids. It deliberately knows nothing about providers or persistence.

In [ ]:
class _ResponseStream(BasicRepr):
    def __init__(self, turn):
        self.turn,self.seq,self.next_index = turn,0,0
        self.message_id,self.message_index,self.text = None,None,''
        self.tool_item_ids,self.streamed_call_ids = {},[]

    def event(self, event_type, **kwargs):
        result = response_event(event_type, self.seq, **kwargs)
        self.seq += 1
        return result

In [ ]:
stream_state = _ResponseStream(provider_turn)
sequenced = [stream_state.event('first'), stream_state.event('second')]
test_eq([json.loads(o.split('data: ', 1)[1])['sequence_number'] for o in sequenced], [0, 1])
_event_types(sequenced)

['first', 'second']

Text begins one assistant item and streams standard content deltas into it.

In [ ]:
@patch
def text_events(self:_ResponseStream, part):
    events = []
    if self.message_id is None:
        self.message_id,self.message_index = _new_id('msg'),self.next_index
        self.next_index += 1
        item = dict(id=self.message_id, type='message', status='in_progress', role='assistant', content=[])
        events.append(self.event('response.output_item.added', output_index=self.message_index, item=item))
        content = dict(type='output_text', text='', annotations=[])
        events.append(self.event('response.content_part.added', item_id=self.message_id,
            output_index=self.message_index, content_index=0, part=content))
    self.text += part.text or ''
    if part.text:
        events.append(self.event('response.output_text.delta', item_id=self.message_id,
            output_index=self.message_index, content_index=0, delta=part.text))
    return events

In [ ]:
text_stream = _ResponseStream(provider_turn)
text_frames = text_stream.text_events(Text('Hello'))
test_eq(_event_types(text_frames), ['response.output_item.added', 'response.content_part.added', 'response.output_text.delta'])
_event_types(text_frames)

['response.output_item.added',
 'response.content_part.added',
 'response.output_text.delta']

Each function call is a complete output item. FastLLM providers already collate tool arguments before yielding `ToolUse`, so the facade can emit the standard delta/done sequence without provider-specific parsing.

In [ ]:
@patch
def tool_events(self:_ResponseStream, part):
    if not part.id: part.id = _new_id('call')
    self.streamed_call_ids.append(part.id)
    item_id,index,args = _new_id('fc'),self.next_index,_arguments(part.arguments)
    self.next_index += 1
    self.tool_item_ids[part.id] = item_id
    item = dict(id=item_id, type='function_call', status='in_progress', call_id=part.id, name=part.name, arguments='')
    events = [self.event('response.output_item.added', output_index=index, item=item)]
    events.append(self.event('response.function_call_arguments.delta', item_id=item_id, output_index=index, delta=args))
    events.append(self.event('response.function_call_arguments.done', item_id=item_id, output_index=index,
        name=part.name, arguments=args))
    item['status'],item['arguments'] = 'completed',args
    events.append(self.event('response.output_item.done', output_index=index, item=item))
    return events

In [ ]:
@patch
def part_events(self:_ResponseStream, part):
    if isinstance(part, Text): return self.text_events(part)
    if isinstance(part, ToolUse) and not part.server: return self.tool_events(part)
    return []

In [ ]:
tool_stream = _ResponseStream(provider_turn)
tool_frames = tool_stream.part_events(ToolUse(name='py', arguments={'code':'2+2'}))
test_eq(_event_types(tool_frames), ['response.output_item.added', 'response.function_call_arguments.delta',
    'response.function_call_arguments.done', 'response.output_item.done'])
_event_types(tool_frames)

['response.output_item.added',
 'response.function_call_arguments.delta',
 'response.function_call_arguments.done',
 'response.output_item.done']

The terminal collation attaches streamed item ids to the final `Completion`, ensuring the terminal Response refers to the same items as its preceding events.

In [ ]:
@patch
def done_events(self:_ResponseStream, comp):
    events = []
    if self.message_id is not None:
        content = dict(type='output_text', text=self.text, annotations=[])
        item = dict(id=self.message_id, type='message', status='completed', role='assistant', content=[content])
        events.append(self.event('response.output_text.done', item_id=self.message_id, output_index=self.message_index,
            content_index=0, text=self.text))
        events.append(self.event('response.content_part.done', item_id=self.message_id, output_index=self.message_index,
            content_index=0, part=content))
        events.append(self.event('response.output_item.done', output_index=self.message_index, item=item))
    for i,tc in enumerate(comp.tool_calls):
        if not tc.id and i < len(self.streamed_call_ids): tc.id = self.streamed_call_ids[i]
        tc._response_item_id = self.tool_item_ids.get(tc.id)
    for part in comp.message.content:
        if isinstance(part, Text): part._response_item_id = self.message_id
    return events

In [ ]:
terminal_comp = Completion('demo', Msg('assistant', [Text('Hello')]), 'stop')
terminal_frames = text_stream.done_events(terminal_comp)
test_eq(_event_types(terminal_frames), ['response.output_text.done', 'response.content_part.done', 'response.output_item.done'])
_event_types(terminal_frames)

['response.output_text.done',
 'response.content_part.done',
 'response.output_item.done']

`events` runs the whole request. A host may start the first provider call itself and pass its `stream`, so a provider that rejects the request can be answered with a real error status before any event is sent. Each provider call streams text and client tool calls as standard events; a completion whose calls are all server tools is answered on the spot and the provider is called again, within the same public response, until a completion needs the client. A batch mixing server and client calls runs the server ones, records their results in the continuation state, and hands the client calls back. The terminal resource and the `finalize` callback see one `Completion`: every step's text in order, and the usage of every provider call summed. `finalize` receives the next state and that `Completion` before the terminal event is emitted, so a host can persist and bill before the client learns the response is complete.


In [ ]:
@patch
async def events(self:AsyncResponses, turn, finalize=None, stream=None):
    state,first = _ResponseStream(turn),turn
    yield state.event('response.created', response=pending_response(turn))
    try:
        texts,usage = [],None
        while True:
            async for part in (stream or await self.call(turn)):
                if isinstance(part, Completion): comp = part
                elif isinstance(part, Status) or (isinstance(part, ToolUse) and part.name in turn.server_tools): continue
                else:
                    for event in state.part_events(part): yield event
            usage,stream = (comp.usage if usage is None else usage + comp.usage),None
            results = await self.server_results(turn, comp)
            if not results or len(results) < len(comp.tool_calls): break
            texts += [p for p in comp.message.content if isinstance(p, Text)]
            turn = self.server_turn(turn, comp, results)
        next_state = self.state(turn, comp, results)
        comp = copy.copy(comp)
        comp.message,comp.usage = Msg('assistant', texts + comp.message.content),usage
        for event in state.done_events(comp): yield event
        response = response_object(first.id, first.body, comp, first.previous.id if first.previous else None, first.created_at)
        if finalize: await finalize(next_state, comp)
        name = 'response.incomplete' if response['status'] == 'incomplete' else 'response.completed'
        yield state.event(name, response=response)
    except asyncio.CancelledError: raise
    except ResponsesError as exc: yield state.event('error', code=exc.code, message=str(exc), param=exc.param)
    except APIError as exc: yield state.event('error', code=exc.code or 'provider_error', message=str(exc), param=None)


`events` lives and dies with its consumer: a host that returns it as an HTTP response sees the generator cancelled when the client disconnects, and the provider call is cut off before `finalize` runs, so the turn's usage is never known. `serve` separates the two. The provider loop runs as a task the facade holds in `tasks`, frames pass through a queue, and the returned iterator only reads. A reader that stops early stops delivery, while the task drains the provider stream to its terminal event and finalizes with the usage the provider reported. `timeout` bounds the task and ends the stream with an `error` frame:

In [ ]:
#| export
@patch
def serve(self:AsyncResponses, turn, finalize=None, stream=None, timeout=None):
    "The frames of `events`, from a task that outlives its reader, so a disconnected client still gets its turn finalized"
    q = asyncio.Queue()
    async def _run():
        try:
            async with asyncio.timeout(timeout):
                async for frame in self.events(turn, finalize=finalize, stream=stream): q.put_nowait(frame)
        except TimeoutError: q.put_nowait(response_event('error', 0, code='timeout', message=f'turn exceeded {timeout}s', param=None))
        finally: q.put_nowait(None)
    t = asyncio.create_task(_run())
    self.tasks.add(t)
    t.add_done_callback(self.tasks.discard)
    async def _frames():
        while (frame := await q.get()) is not None: yield frame
    return _frames()

## Provider turns

A real GPT-5.6 Luna turn demonstrates the facade's intended tool-loop boundary. Cachy records these HTTP calls for repeatable lesson execution; production code does not enable it.

In [ ]:
enable_cachy(doms=['api.openai.com'])

In [ ]:
py_tool = dict(type='function', name='py', description='Run Python code',
    parameters=dict(type='object', properties={'code':{'type':'string'}}, required=['code'], additionalProperties=False))
live_body = dict(model='codex/gpt-5.6-luna', input='Use py to calculate 12345*6789. Do not calculate it yourself.',
    instructions='Return one py tool call.', tools=[py_tool], tool_choice='required', parallel_tool_calls=True)
live_body

{'model': 'codex/gpt-5.6-luna',
 'input': 'Use py to calculate 12345*6789. Do not calculate it yourself.',
 'instructions': 'Return one py tool call.',
 'tools': [{'type': 'function',
   'name': 'py',
   'description': 'Run Python code',
   'parameters': {'type': 'object',
    'properties': {'code': {'type': 'string'}},
    'required': ['code'],
    'additionalProperties': False}}],
 'tool_choice': 'required',
 'parallel_tool_calls': True}

In [ ]:
#| hide
live_final = []
async def capture_live(state, comp): live_final.append((state, comp))

The streamed turn ends with a normal `response.completed` event carrying the tool call, while the finalizer retains the next continuation state, which round-trips through its dict form with the provider message intact.

In [ ]:
live_turn = responses.prepare(live_body)
live_frames = [frame async for frame in responses.events(live_turn, finalize=capture_live)]
live_state,live_comp = live_final[0]
test_eq((live_comp.finish_reason, live_comp.tool_calls[0].name), ('tool_calls', 'py'))
test_eq(live_frames[-1].splitlines()[0], 'event: response.completed')
test_eq(ResponseState.from_dict(live_state.to_dict()).history[-1].raw, live_state.history[-1].raw)
live_comp.tool_calls

[ToolUse(raw={'id': 'fc_0e6991daa82ca74c016a955a67237087d0b244036e42d2299c', 'type': 'function_call', 'status': 'completed'}, cache_control=None, id='call_LPcF1KKvQIvsv8xutE4HPGmL', name='py', arguments={'code': '12345*6789'}, server=False, text=None)]

The client returns the tool output with the public response id. FastLLM uses the retained upstream id when available and still returns a fresh public Response resource.

In [ ]:
live_call = live_comp.tool_calls[0]
continued_body = dict(model=live_body['model'], tools=[py_tool], tool_choice='none', previous_response_id=live_state.id)
continued_body['input'] = [dict(type='function_call_output', call_id=live_call.id, output='83810205')]
continued_body

{'model': 'codex/gpt-5.6-luna',
 'tools': [{'type': 'function',
   'name': 'py',
   'description': 'Run Python code',
   'parameters': {'type': 'object',
    'properties': {'code': {'type': 'string'}},
    'required': ['code'],
    'additionalProperties': False}}],
 'tool_choice': 'none',
 'previous_response_id': 'resp_d2f3def9442e60349ddc8ce3203d1442',
 'input': [{'type': 'function_call_output',
   'call_id': 'call_LPcF1KKvQIvsv8xutE4HPGmL',
   'output': '83810205'}]}

In [ ]:
#| hide
continued_final = []
async def capture_continued(state, comp): continued_final.append((state, comp))

Because Codex has no usable provider continuation id, this second streamed turn replays FastLLM's canonical history and still presents the same public `previous_response_id` contract. A wire `function_call_output` carries no function name, so `prepare` restores it from the matching call in history: providers such as Gemini require it on the tool result.

In [ ]:
continued_turn = responses.prepare(continued_body, live_state)
test_eq(first(p for m in continued_turn.provider_messages for p in m.content if isinstance(p, ToolResult)).name, 'py')
continued_frames = [frame async for frame in responses.events(continued_turn, finalize=capture_continued)]
continued_state,continued_comp = continued_final[0]
assert '83810205' in continued_comp.message.text.replace(',', '')
test_eq(continued_turn.previous.id, live_state.id)
continued_comp.message

**Msg**

- role: `assistant`

<contents>

**Text** (`text`)

83,810,205

::: details

- raw: `None`
- citations: `[]`

:::

</contents>

A live turn shows the loop end to end. GPT-5.6 Luna gets `lookup` as a server tool and nothing else, so the request contains its call and result invisibly: the stream carries no function-call items, the continuation history holds the internal call and answer, and the usage is the sum of both provider calls.

In [ ]:
lookup_final = []
async def capture_lookup(state, comp): lookup_final.append((state, comp))
lookup_body = dict(model='openai/gpt-5.6-luna', input="What does 'gorp' mean according to the glossary? Use lookup, then answer in one sentence.", tools=[])
lookup_frames = [frame async for frame in responses.events(responses.prepare(lookup_body, server_tools=[lookup]), finalize=capture_lookup)]
lookup_state,lookup_comp = lookup_final[0]
test_eq([m.role for m in lookup_state.history], ['user', 'assistant', 'tool', 'assistant'])
assert 'function_call' not in ''.join(lookup_frames)
assert 'trail mix' in lookup_comp.message.text.lower(), lookup_comp.message.text
lookup_comp.message.text, lookup_comp.usage.prompt_tokens

A served stream outlives its reader. The consumer here takes two frames and closes, as a disconnected client would. The task carries on to the provider's terminal event, and the finalizer still runs, once, with the usage of the whole turn:

In [ ]:
served_final = []
async def capture_served(state, comp): served_final.append((state, comp))
frames = responses.serve(responses.prepare(live_body), finalize=capture_served)
first_two = [await anext(frames), await anext(frames)]
await frames.aclose()
await asyncio.gather(*responses.tasks)
test_eq(len(served_final), 1)
test_eq(served_final[0][1].usage.total_tokens, live_comp.usage.total_tokens)
first_two[0].splitlines()[0]

## Export

In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()